In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [41]:
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.linear_model import Ridge, Lasso, ElasticNet, BayesianRidge
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from scipy.stats import pearsonr
from sklearn.metrics import mean_squared_error
import spacy

data_a    = '/content/drive/MyDrive/Q3_mlp'
data_b = '/content/drive/MyDrive/data'
BIOGPT_CSV   = f'{data_a}/biogpt.csv'
CLINICAL_CSV = f'{data_a}/BiomedClinicalBERT_winning_pipeline_results.csv'
EXP0_CSV     = f'{data_a}/exp0.csv'
EXP1_CSV     = f'{data_a}/exp1_test_results.csv'
N_SPLITS     = 5
EPOCHS       = 150
LR           = 1e-3
HIDDEN_DIMS  = [32]
DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [42]:
def load_csv(path):
    df = pd.read_csv(path)
    df.columns = [c.lower().replace('pubmed_id', 'pmid') for c in df.columns]
    df['pmid'] = df['pmid'].astype(str)
    return df

exp0     = load_csv(EXP0_CSV)
exp1     = load_csv(EXP1_CSV)
biogpt   = load_csv(BIOGPT_CSV)
clinical = load_csv(CLINICAL_CSV)

base = exp0[['pmid']].copy()
base['correct_exp0']     = exp0['correct'].values
base['correct_exp1']     = exp1.set_index('pmid').loc[base['pmid']]['correct'].values
base['correct_biogpt']   = biogpt.set_index('pmid').loc[base['pmid']]['correct'].values
base['correct_clinical'] = base['pmid'].map(dict(zip(clinical['pmid'], clinical['correct'])))
base['difficulty']       = base[['correct_exp0','correct_exp1','correct_biogpt','correct_clinical']].mean(axis=1)

In [43]:
data_map = {}
for fold in range(10):
    for split in ['train_set.json', 'dev_set.json']:
        with open(f'{data_b}/pqal_fold{fold}/{split}', 'r') as f:
            raw = json.load(f)
        for pmid, item in raw.items():
            data_map[str(pmid)] = item

with open(f'{data_b}/test_set.json', 'r') as f:
    for pmid, item in json.load(f).items():
        data_map[str(pmid)] = item

base['raw'] = base['pmid'].map(data_map)
base = base.dropna(subset=['raw']).reset_index(drop=True)
base['question']           = base['raw'].map(lambda x: x['QUESTION'])
base['abstract']           = base['raw'].map(lambda x: ' '.join(x['CONTEXTS']))
base['meshes']             = base['raw'].map(lambda x: x.get('MESHES', []))
base['year']               = base['raw'].map(lambda x: int(x['YEAR']) if x.get('YEAR') else 2000)
base['labels']             = base['raw'].map(lambda x: x.get('LABELS', []))
base['reasoning_required'] = base['raw'].map(lambda x: 1 if x.get('reasoning_required_pred') == 'yes' else 0)
base['reasoning_free']     = base['raw'].map(lambda x: 1 if x.get('reasoning_free_pred') == 'yes' else 0)
base = base.drop(columns=['raw'])
print(f"Linked samples : {len(base)}")

Linked samples : 500


In [44]:
nlp = spacy.load('en_core_web_sm')

def extract_features(row):
    doc_q = nlp(row['question'])
    doc_a = nlp(row['abstract'][:1000])

    q_tokens = set(t.lemma_.lower() for t in doc_q if not t.is_stop and t.is_alpha)
    a_tokens = set(t.lemma_.lower() for t in doc_a if not t.is_stop and t.is_alpha)
    q_ents   = set(e.text.lower() for e in doc_q.ents)
    a_ents   = set(e.text.lower() for e in doc_a.ents)

    return {
        'q_length'           : len(list(doc_q)),
        'q_num_entities'     : len(doc_q.ents),
        'q_has_negation'     : int(any(t.dep_ == 'neg' for t in doc_q)),
        'q_has_comparison'   : int(any(w in row['question'].lower() for w in ['versus','compared','vs','than'])),
        'q_avg_word_len'     : np.mean([len(t.text) for t in doc_q if t.is_alpha]) if any(t.is_alpha for t in doc_q) else 0,
        'q_num_nouns'        : sum(1 for t in doc_q if t.pos_ == 'NOUN'),
        'q_num_verbs'        : sum(1 for t in doc_q if t.pos_ == 'VERB'),
        'q_has_whether'      : int('whether' in row['question'].lower()),
        'q_has_population'   : int(any(w in row['question'].lower() for w in ['patient','children','adult','elderly','women','men'])),
        'q_has_outcome'      : int(any(w in row['question'].lower() for w in ['survival','mortality','risk','rate','efficacy'])),
        'a_length'           : len(list(doc_a)),
        'a_num_entities'     : len(doc_a.ents),
        'a_has_negation'     : int(any(t.dep_ == 'neg' for t in doc_a)),
        'a_has_significance' : int(any(w in row['abstract'].lower() for w in ['significant','p <','p=','p value'])),
        'a_has_results'      : int('RESULTS' in row['labels']),
        'a_has_methods'      : int('METHODS' in row['labels']),
        'token_overlap'      : len(q_tokens & a_tokens) / (len(q_tokens) + 1),
        'entity_overlap'     : len(q_ents & a_ents),
        'q_a_len_ratio'      : len(list(doc_q)) / (len(list(doc_a)) + 1),
        'a_answers_directly' : int(any(w in row['abstract'].lower() for w in ['yes','no','conclude','conclusion','suggest'])),
        'num_meshes'         : len(row['meshes']),
        'year'               : row['year'],
        'reasoning_required' : row['reasoning_required'],
        'reasoning_free'     : row['reasoning_free'],
        'reasoning_mismatch' : int(row['reasoning_required'] != row['reasoning_free']),
    }
feature_df = pd.DataFrame([extract_features(row) for _, row in base.iterrows()])
X = feature_df.values
y = base['difficulty'].values

print(f"Feature matrix : {X.shape}")

Feature matrix : (500, 25)


In [45]:
class RegressionMLP(nn.Module):
    def __init__(self, input_dim, hidden_dims):
        super().__init__()
        layers = []
        prev   = input_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(0.5)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(1)

def train_mlp(X_train, y_train, X_val):
    X_tr_t = torch.FloatTensor(X_train).to(DEVICE)
    y_tr_t = torch.FloatTensor(y_train).to(DEVICE)
    X_vl_t = torch.FloatTensor(X_val).to(DEVICE)

    mlp       = RegressionMLP(X_train.shape[1], HIDDEN_DIMS).to(DEVICE)
    optimizer = torch.optim.Adam(mlp.parameters(), lr=LR, weight_decay=1e-3)
    criterion = nn.MSELoss()
    loader    = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=16, shuffle=True)

    mlp.train()
    for epoch in range(EPOCHS):
        for xb, yb in loader:
            optimizer.zero_grad()
            criterion(mlp(xb), yb).backward()
            optimizer.step()

    mlp.eval()
    with torch.no_grad():
        return mlp(X_vl_t).cpu().numpy()

In [54]:
models = {
    'Ridge'        : lambda: Ridge(alpha=1.0),
    'Lasso'        : lambda: Lasso(alpha=0.01),
    'BayesianRidge': lambda: BayesianRidge(),}

kf      = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
results = {name: {'mse': [], 'pearson': []} for name in list(models.keys()) + ['MLP']}

for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    scaler  = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val   = scaler.transform(X_val)

    for name, model_fn in models.items():
        model = model_fn()
        model.fit(X_train, y_train)
        pred = model.predict(X_val)
        results[name]['mse'].append(mean_squared_error(y_val, pred))
        results[name]['pearson'].append(pearsonr(y_val, pred)[0])

    mlp_pred = train_mlp(X_train, y_train, X_val)
    results['MLP']['mse'].append(mean_squared_error(y_val, mlp_pred))
    results['MLP']['pearson'].append(pearsonr(y_val, mlp_pred)[0])

    print(f"Fold {fold+1} done")

Fold 1 done
Fold 2 done
Fold 3 done
Fold 4 done
Fold 5 done


In [56]:
summary = pd.DataFrame({
    'MSE'      : [f"{np.mean(s['mse']):.3f} ± {np.std(s['mse']):.3f}" for s in results.values()],
    'Pearson r': [f"{np.mean(s['pearson']):.3f} ± {np.std(s['pearson']):.3f}" for s in results.values()],
}, index=list(results.keys()))
summary

,MSE,Pearson r
Ridge,0.080 ± 0.010,0.581 ± 0.058
Lasso,0.076 ± 0.009,0.603 ± 0.056
BayesianRidge,0.079 ± 0.010,0.584 ± 0.055
MLP,0.092 ± 0.011,0.506 ± 0.058


In [57]:
pd.Series(Lasso(alpha=0.01).fit(StandardScaler().fit_transform(X), y).coef_, index=feature_df.columns).abs().nlargest(5)

,0
reasoning_free,0.125940
reasoning_required,0.089405
reasoning_mismatch,0.063279
q_has_population,0.012606
q_num_verbs,0.005520
